# *Game over or High Score?*

#### This project analyzes historical video game sales for Ice, an online store that sells games worldwide. Using open data on user and critic reviews, genres, platforms (such as Xbox or PlayStation) and regional sales, the goal is to identify the patterns that make a game successful. These insights will help spot potential best sellers and plan advertising campaigns.

#### The data goes up to 2016. The analysis is set in December 2016, with the aim of planning the campaign for 2017.

## *1. Import and Overview*

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from scipy import stats

In [2]:
games_df = pd.read_csv('datasets\games.csv')
display(games_df.sample(5))
print()
games_df.info()

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
14665,Saihai no Yukue,DS,2008.0,Strategy,0.00,0.00,0.03,0.00,NaN,NaN,NaN
364,The Getaway,PS2,2002.0,Action,1.23,1.77,0.05,0.49,72.0,6.8,M
9507,007: Quantum of Solace,DS,2008.0,Action,0.11,0.01,0.00,0.01,65.0,tbd,T
8376,World Tour Soccer 2006,PS2,2004.0,Sports,0.08,0.06,0.00,0.02,56.0,6,E
15421,Aria: The Natural ~Tooi Yume no Mirage~,PS2,2006.0,Adventure,0.00,0.00,0.02,0.00,NaN,NaN,NaN



<class 'pandas.DataFrame'>
RangeIndex: 16715 entries, 0 to 16714
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16713 non-null  str    
 1   Platform         16715 non-null  str    
 2   Year_of_Release  16446 non-null  float64
 3   Genre            16713 non-null  str    
 4   NA_sales         16715 non-null  float64
 5   EU_sales         16715 non-null  float64
 6   JP_sales         16715 non-null  float64
 7   Other_sales      16715 non-null  float64
 8   Critic_Score     8137 non-null   float64
 9   User_Score       10014 non-null  str    
 10  Rating           9949 non-null   str    
dtypes: float64(6), str(5)
memory usage: 2.0 MB


## *2. Data Cleaning*

#### Our initial check showed some issues that need to be fixed:
1. **Standardize names and text values:** convert column names and text values to lowercase and remove extra spaces.
2. **Check for duplicates:** identify and remove duplicated rows, if any.
3. **Handle missing values:** count missing values per column and decide case by case. Drop rows only where very few values are missing; keep large gaps as NaN and explain why.
4. **Handle "tbd" in user_score:** treat "tbd" as a missing value (NaN), since the score has not been determined yet.
5. **Convert data types:**
   1. `year_of_release`: float to int, since years have no decimals (rows with missing years are removed first).
   2. `user_score`: text to float, so it can be used in calculations.
   3. Sales columns stay as float, since values are in millions and decimals matter.
6. **Calculate total sales:** add a column with the sum of sales across all regions.

In [3]:
# Standardizing column names and text values
games_df.columns = games_df.columns.str.lower()

for column in ['name', 'platform', 'genre', 'rating']:
    games_df[column] = games_df[column].str.lower().str.strip()

In [4]:
# Checking unique values

for column in ['name', 'platform', 'genre', 'rating']:
    print(f'{column}: {games_df[column].nunique()} unique values')
    print(games_df[column].dropna().sort_values().unique())
    print()

name: 11559 unique values
<ArrowStringArray>
[                                  ''98 koshien',
                    '.hack//g.u. vol.1//rebirth',
                  '.hack//g.u. vol.2//reminisce',
       '.hack//g.u. vol.2//reminisce (jp sales)',
                 '.hack//g.u. vol.3//redemption',
                       '.hack//infection part 1',
                                   '.hack//link',
                        '.hack//mutation part 2',
                        '.hack//outbreak part 3',
   '.hack//quarantine part 4: the final chapter',
 ...
                                          'zoop',
                                          'zubo',
                                 'zumba fitness',
                               'zumba fitness 2',
                            'zumba fitness core',
                            'zumba fitness rush',
                    'zumba fitness: world party',
                                        'zwei!!',
 'zyuden sentai kyoryuger: game de gaburincho!!',


In [5]:
# Checking duplicates based on name and platform

games_df[games_df.duplicated(subset=['name', 'platform'], keep=False)]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
604,madden nfl 13,ps3,2012.0,sports,2.11,0.22,0.00,0.23,83.0,5.5,e
659,NaN,gen,1993.0,NaN,1.78,0.53,0.00,0.08,NaN,NaN,NaN
1190,need for speed: most wanted,x360,2012.0,racing,0.62,0.78,0.01,0.15,83.0,8.5,t
1591,need for speed: most wanted,x360,2005.0,racing,1.00,0.13,0.02,0.10,83.0,8.5,t
1745,sonic the hedgehog,ps3,2006.0,platform,0.41,0.06,0.04,0.66,43.0,4.1,e10+
4127,sonic the hedgehog,ps3,NaN,platform,0.00,0.48,0.00,0.00,43.0,4.1,e10+
5972,need for speed: most wanted,pc,2005.0,racing,0.02,0.23,0.00,0.04,82.0,8.5,t
11715,need for speed: most wanted,pc,2012.0,racing,0.00,0.06,0.00,0.02,82.0,8.5,t
14244,NaN,gen,1993.0,NaN,0.00,0.00,0.03,0.00,NaN,NaN,NaN
16230,madden nfl 13,ps3,2012.0,sports,0.00,0.01,0.00,0.00,83.0,5.5,e


In [6]:
sales_columns = ['na_sales', 'eu_sales', 'jp_sales', 'other_sales']

games_df.loc[604, sales_columns] += games_df.loc[16230, sales_columns]
games_df.loc[1745, sales_columns] += games_df.loc[4127, sales_columns]

games_df = games_df.drop(index=[16230, 4127]).reset_index(drop=True)

#### After checking the duplicates, I made the following decisions:
* **Need for Speed: Most Wanted was kept:** the rows have different release years (2005 and 2012), so they are two different games with the same name.
* **The two rows without a name were not treated here:** they also have no genre, so they are removed in the missing values step.
* **Madden NFL 13 and Sonic the Hedgehog were merged:** their sales were summed into one row, and the extra row was removed.

In [7]:
# Fixing missing values

games_df = games_df.dropna(subset=['name', 'genre', 'year_of_release']) # Removed because these rows represent around 1.6% of the data

games_df['rating'] = games_df['rating'].fillna('unknown')

In [8]:
# Fixing column data types

games_df['year_of_release'] = games_df['year_of_release'].astype(int)
games_df['user_score'] = pd.to_numeric(games_df['user_score'], errors='coerce') # Also converts "tbd" to NaN

In [9]:
# Adding total_sales column

games_df.insert(8, 'total_sales', games_df[sales_columns].sum(axis=1))

In [10]:
display(games_df.sample(5))
print()
games_df.info()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,total_sales,critic_score,user_score,rating
889,tom clancy's ghost recon,ps2,2002,shooter,1.42,0.40,0.0,0.12,1.94,63.0,8.3,m
6647,disney sing it: pop hits,ps3,2009,misc,0.17,0.05,0.0,0.03,0.25,NaN,NaN,e
14977,monster mayhem: build and battle,ds,2009,simulation,0.02,0.00,0.0,0.00,0.02,NaN,NaN,e
3369,avatar: the game,psp,2009,action,0.22,0.24,0.0,0.14,0.60,NaN,NaN,unknown
5329,enemy territory: quake wars,ps3,2008,shooter,0.23,0.07,0.0,0.04,0.34,60.0,6.4,t



<class 'pandas.DataFrame'>
Index: 16443 entries, 0 to 16712
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16443 non-null  str    
 1   platform         16443 non-null  str    
 2   year_of_release  16443 non-null  int64  
 3   genre            16443 non-null  str    
 4   na_sales         16443 non-null  float64
 5   eu_sales         16443 non-null  float64
 6   jp_sales         16443 non-null  float64
 7   other_sales      16443 non-null  float64
 8   total_sales      16443 non-null  float64
 9   critic_score     7982 non-null   float64
 10  user_score       7462 non-null   float64
 11  rating           16443 non-null  str    
dtypes: float64(7), int64(1), str(4)
memory usage: 2.2 MB


### *Conclusion*
* **Standardization:** column names and text values converted to lowercase, with extra spaces removed.
* **Duplicates:** Madden NFL 13 and Sonic the Hedgehog (PS3) were split records, so their sales were summed into one row.
* **Missing values:**
   * Rows without name, genre or year were removed (about 1.6% of the data).
   * Missing ratings became "unknown", likely games released before the ESRB or outside North America.
   * Critic and user scores were kept as NaN, since about half are missing and filling them would distort the analysis. These are likely older or smaller games with few reviews.
   * "tbd" means not enough user reviews yet, so it was treated as NaN.
* **Data types:**
   * `year_of_release`: float to int, since years have no decimals.
   * `user_score`: text to float, so it can be used in calculations.
* **New column:** `total_sales`, the sum of all regional sales.